# Stage 1 - Dense-UNet Lung Segmentation (Fast-Track Kaggle)
## Anatomical Inductive Bias Gate for AG-MTCNet
---
This notebook executes the Stage 1 training of **Dense-UNet** for anatomical lung segmentation based on:

> Dubey et al., *Deep learning-based anatomical segmentation of chest radiographs for computer-aided diagnosis*, BMC Infectious Diseases, 2026.

### Key Architectural and Safety Innovations
1. **Contracting Path**: DenseNet-121 backbone providing dense feature reuse and direct gradient flow (single source of truth: `src/models/dense_unet.py`).
2. **Numerical Safety**: Per-sample Soft Dice Loss + BCE with FP32 clamping (prevents AMP FP16 NaNs).
3. **Clinical Fail-Safe Protocol**: Automated white-out lung and massive effusion safeguard (`min_lung_coverage < 0.18` or severe hemithoracic asymmetry -> gate bypassed).
4. **GPU-Accelerated Tensor Gating**: `apply_dilated_soft_mask_tensor` runs inside the PyTorch pipeline with zero CPU-GPU roundtrip overhead.
5. **Contrast Invariance**: CLAHE jitter (`clipLimit` 1.5-3.5, random tile grid) forces the network to learn lung anatomy independent of hospital-specific windowing.
6. **Multi-Environment**: Auto-clones the GitHub repository on Kaggle, reuses the local checkout otherwise, discovers real lung masks under `/kaggle/input`, and falls back to a synthetic thoracic generator so the notebook always completes end-to-end.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Cross-environment bootstrap (Kaggle clone or local sys.path)
# ---------------------------------------------------------------------------
import os
import sys
import time
import random
import shlex
import subprocess
from pathlib import Path

import torch

REPO_URL = "https://github.com/philipsgn/LungRay.git"
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 15
NUM_WORKERS = int(os.environ.get("LUNGRAY_NUM_WORKERS", "2"))
IS_KAGGLE = Path("/kaggle/input").exists() or Path("/kaggle/working").exists()

# Refuse an expensive Kaggle CPU run immediately; local CPU dry-runs remain supported.
if IS_KAGGLE and not torch.cuda.is_available():
    raise SystemError(
        "Kaggle GPU is required for the 15-epoch Dense-UNet run. "
        "Enable a Tesla T4 x2 or P100 accelerator, then restart the session."
    )


def run(cmd, check=True, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Command failed (rc={rc})")
    return rc


if IS_KAGGLE:
    WORK = Path("/kaggle/working")
    WORK.mkdir(parents=True, exist_ok=True)
    REPO = Path("/tmp/LungRay")
    if not (REPO / "src" / "models" / "dense_unet.py").exists():
        if REPO.exists():
            import shutil
            shutil.rmtree(REPO)
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
else:
    here = Path.cwd().resolve()
    REPO = next(
        (p for p in [here, *here.parents] if (p / "src" / "models" / "dense_unet.py").exists()),
        here,
    )
    sys.path.insert(0, str(REPO))

print("[OK] Repository root:", REPO, flush=True)

for module_name, package_name in (
    ("cv2", "opencv-python-headless"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
):
    try:
        __import__(module_name)
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", package_name])

import cv2
import numpy as np
from torch.utils.data import DataLoader, Dataset, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(f"[*] Execution hardware: {device} (CUDA available: {torch.cuda.is_available()})", flush=True)


## 1. Data discovery
The notebook recursively searches Kaggle inputs for paired radiographs and masks under `data_512`, `images/` + `masks/`, `cxr_png/` + `manualmask/`, and nested left/right mask layouts. At least eight valid pairs are required to mark the run as real-data mode. On Kaggle, fewer than eight pairs prints `[WARNING] Real CXR masks not found!` and lists mounted input entries. Synthetic fallback is clearly tagged for pipeline smoke testing and must not be reported as real anatomical validation.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Real lung-mask discovery with safe synthetic fallback
# ---------------------------------------------------------------------------
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def iter_images(root, max_depth=6, limit=20000):
    root = Path(root)
    if not root.exists():
        return
    base = len(root.parts)
    found = 0
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base > max_depth:
            dirnames[:] = []
            continue
        for name in filenames:
            if Path(name).suffix.lower() in IMAGE_EXTS:
                yield here / name
                found += 1
                if found >= limit:
                    return


_MASK_SUFFIXES = (
    "_leftmask", "_rightmask", "-leftmask", "-rightmask",
    "_left", "_right", "-left", "-right",
    "_mask", "-mask", "_masks", "-masks",
    "_lungmask", "-lungmask",
)


def mask_stem(path):
    stem = path.stem
    low = stem.lower()
    for suffix in _MASK_SUFFIXES:
        if low.endswith(suffix) and len(stem) > len(suffix):
            return stem[: -len(suffix)]
    return stem


def discover_mask_pairs():
    """Discover paired CXR/mask files across Kaggle and local folder variants."""
    input_root = Path("/kaggle/input")
    roots = []
    if input_root.exists():
        roots.append(input_root)
        roots.extend(sorted(p for p in input_root.rglob("data_512") if p.is_dir()))
    local_root = REPO / "dataset"
    if local_root.exists():
        roots.append(local_root)
    working_root = Path("/kaggle/working") / "data_512"
    if working_root.exists():
        roots.append(working_root)

    image_dir_names = {"images", "image", "images_512", "cxr_png", "cxr", "scans", "xray"}
    mask_dir_names = {"masks", "mask", "masks_512", "manualmask", "leftmask", "rightmask", "lung_mask"}
    seen_roots = set()
    for root in roots:
        root = root.resolve()
        if root in seen_roots:
            continue
        seen_roots.add(root)
        images, masks = {}, {}
        for path in iter_images(root, max_depth=8, limit=50000):
            parts_lower = [part.lower() for part in path.parts]
            # Ignore checkpoint and hidden folders
            if any(p.startswith(".") for p in parts_lower):
                continue
            parent_names = [part.lower() for part in path.parent.parts]
            in_mask_tree = any(
                name in mask_dir_names or name.startswith("mask")
                for name in parent_names
            )
            stem_has_mask = "mask" in path.stem.lower()
            if in_mask_tree or stem_has_mask:
                masks.setdefault(mask_stem(path), []).append(path)
            else:
                images.setdefault(path.stem, path)

        pairs = [
            (str(images[stem]), [str(m) for m in mask_paths], stem)
            for stem, mask_paths in masks.items()
            if stem in images
        ]
        if len(pairs) >= 8:
            return pairs, root
    return [], None


def letterbox_gray(image, size):
    h, w = image.shape[:2]
    scale = size / max(h, w)
    nh = max(1, int(round(h * scale)))
    nw = max(1, int(round(w * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)
    top = (size - nh) // 2
    left = (size - nw) // 2
    return cv2.copyMakeBorder(
        resized, top, size - nh - top, left, size - nw - left,
        cv2.BORDER_CONSTANT, value=0,
    )


def clahe_contrast_jitter(image_u8, rng, p=0.50):
    """Randomized CLAHE: hospital-windowing invariance augmentation."""
    if rng.random() > p:
        return image_u8
    clip_limit = float(rng.uniform(1.5, 3.5))
    tile = int(rng.choice([4, 8, 16]))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))
    return clahe.apply(image_u8)


def synthetic_pair(idx, img_size):
    rng = np.random.default_rng(1234 + idx)
    h = w = img_size
    image = np.full((h, w), int(rng.integers(25, 55)), dtype=np.uint8)
    cv2.ellipse(image, (w // 2, h // 2), (int(w * 0.42), int(h * 0.46)), 0, 0, 360, 165, -1)
    mask = np.zeros((h, w), dtype=np.uint8)
    shift = int(rng.integers(1, 6))
    cv2.ellipse(mask, (int(w * 0.32) + shift, int(h * 0.48)), (int(w * 0.14), int(h * 0.30)), 0, 0, 360, 255, -1)
    cv2.ellipse(mask, (int(w * 0.68) - shift, int(h * 0.48)), (int(w * 0.14), int(h * 0.30)), 0, 0, 360, 255, -1)
    image[mask > 0] = int(rng.integers(55, 95))
    image = cv2.add(image, rng.integers(0, 16, (h, w), dtype=np.uint8))
    return image, mask


class CXRSegmentationDataset(Dataset):
    """Dual-mode CXR segmentation dataset: real lung masks or synthetic fallback."""

    def __init__(self, pairs=None, num_samples=96, img_size=256, is_train=True, seed=SEED):
        self.pairs = pairs
        self.num_samples = len(pairs) if pairs else int(num_samples)
        self.img_size = int(img_size)
        self.is_train = bool(is_train)
        self.seed = int(seed)

    def __len__(self):
        return self.num_samples

    def _load_real(self, img_path, mask_paths):
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise IOError(f"Unreadable image: {img_path}")
        mask = np.zeros_like(image)
        for mask_path in mask_paths:
            m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            if m.shape[:2] != image.shape[:2]:
                m = cv2.resize(m, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            mask = cv2.max(mask, m)
        return image, mask

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + idx * 7919)
        if self.pairs:
            img_path, mask_paths, _ = self.pairs[idx]
            image, mask = self._load_real(img_path, mask_paths)
        else:
            image, mask = synthetic_pair(idx, self.img_size)

        image = letterbox_gray(image, self.img_size)
        mask = letterbox_gray(mask, self.img_size)
        if self.is_train:
            if rng.random() < 0.5:
                image = cv2.flip(image, 1)
                mask = cv2.flip(mask, 1)
            image = clahe_contrast_jitter(image, rng, p=0.5)

        image_rgb = np.stack([image, image, image], axis=-1).astype(np.float32) / 255.0
        image_norm = (image_rgb - IMAGENET_MEAN) / IMAGENET_STD
        mask_f = (mask > 127).astype(np.float32)[None, ...]
        return torch.from_numpy(image_norm.transpose(2, 0, 1)), torch.from_numpy(mask_f)


PAIRS, PAIR_ROOT = discover_mask_pairs()
USE_REAL = len(PAIRS) >= 8
print(f"[*] Real lung-mask pairs: {len(PAIRS)}" + (f" (root: {PAIR_ROOT})" if PAIR_ROOT else ""), flush=True)
print("[*] Data mode:", "REAL lung masks" if USE_REAL else "SYNTHETIC anatomical fallback", flush=True)
if IS_KAGGLE and len(PAIRS) < 8:
    input_root = Path("/kaggle/input")
    mounted = sorted(str(p) for p in input_root.iterdir()) if input_root.exists() else []
    print("[WARNING] Real CXR masks not found! Fewer than 8 valid image-mask pairs were discovered.", flush=True)
    print("[*] Current /kaggle/input entries:", mounted or "<empty>", flush=True)
    print("[WARNING] Synthetic fallback is for pipeline smoke testing only; its metrics are not anatomical validation.", flush=True)


In [3]:
# ---------------------------------------------------------------------------
# 2. Deterministic train/validation partition and DataLoaders
# ---------------------------------------------------------------------------
if USE_REAL:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(PAIRS))
    n_val = max(8, int(round(0.20 * len(PAIRS))))
    val_ids = set(order[:n_val].tolist())
    train_ids = [i for i in range(len(PAIRS)) if i not in val_ids]
    train_ds = Subset(CXRSegmentationDataset(PAIRS, img_size=IMG_SIZE, is_train=True), train_ids)
    val_ds = Subset(CXRSegmentationDataset(PAIRS, img_size=IMG_SIZE, is_train=False), sorted(val_ids))
else:
    train_ds = CXRSegmentationDataset(None, num_samples=96, img_size=IMG_SIZE, is_train=True)
    val_ds = CXRSegmentationDataset(None, num_samples=24, img_size=IMG_SIZE, is_train=False)

pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, drop_last=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, drop_last=False,
)
print(f"[*] Train: {len(train_ds)} images -> {len(train_loader)} batches | Val: {len(val_ds)} images -> {len(val_loader)} batches")


[*] Train: 96 images -> 12 batches | Val: 24 images -> 3 batches


In [4]:
# ---------------------------------------------------------------------------
# 3. Architecture and loss (single source of truth: src/models/dense_unet.py)
# ---------------------------------------------------------------------------
from src.models.dense_unet import (
    CombinedSegmentationLoss,
    DenseUNet,
    apply_dilated_soft_mask_tensor,
    compute_segmentation_metrics,
)

model = DenseUNet(pretrained=False, in_channels=3).to(device)
criterion = CombinedSegmentationLoss(alpha=0.50)
print(f"[*] Dense-UNet instantiated: {sum(p.numel() for p in model.parameters()):,} parameters.")
print("[*] Loss: 0.50 * BCE + 0.50 * per-sample Soft Dice (FP32 clamped, AMP-safe).")

# GPU tensor gating + clinical fail-safe smoke check before training.
with torch.no_grad():
    probe = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    probe_mask = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE, device=device)
    probe_mask[0, 0, IMG_SIZE // 4:IMG_SIZE // 2, IMG_SIZE // 5:IMG_SIZE // 3] = 1.0
    gated_probe = apply_dilated_soft_mask_tensor(probe, probe_mask)
assert gated_probe.shape == probe.shape
assert torch.allclose(gated_probe[1], probe[1]), "White-out fail-safe must preserve the full radiograph."
print("[OK] GPU-dilated soft gate + White-out Lung Fail-Safe verified.")


[*] Dense-UNet instantiated: 17,058,337 parameters.
[*] Loss: 0.50 * BCE + 0.50 * per-sample Soft Dice (FP32 clamped, AMP-safe).
[OK] GPU-dilated soft gate + White-out Lung Fail-Safe verified.


In [ ]:
# ---------------------------------------------------------------------------
# 4. Live training and validation loop with AMP and cosine LR scheduling
# ---------------------------------------------------------------------------
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


def train_one_epoch(model, dataloader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    seen = 0
    for images, masks in dataloader:
        images = images.to(device, non_blocking=(device.type == "cuda"))
        masks = masks.to(device, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            probabilities = model(images)
        # The loss is deliberately evaluated in FP32 outside autocast.
        loss = criterion(probabilities.float(), masks.float())
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss detected: {loss.detach().item()}")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=5.0, error_if_nonfinite=True
        )
        if not torch.isfinite(grad_norm):
            raise FloatingPointError("Non-finite gradient norm detected after unscaling.")
        scaler.step(optimizer)
        scaler.update()
        batch_size = int(images.shape[0])
        running_loss += float(loss.detach()) * batch_size
        seen += batch_size
    return running_loss / max(1, seen)


@torch.no_grad()
def evaluate_epoch(model, dataloader, criterion):
    model.eval()
    running_loss, dice_total, iou_total, count = 0.0, 0.0, 0.0, 0
    for images, masks in dataloader:
        images = images.to(device, non_blocking=(device.type == "cuda"))
        masks = masks.to(device, non_blocking=(device.type == "cuda"))
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            probabilities = model(images)
        loss = criterion(probabilities.float(), masks.float())
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite validation loss detected: {loss.detach().item()}")
        running_loss += float(loss) * images.shape[0]
        pred_bin = (probabilities.squeeze(1).float().cpu().numpy() >= 0.50).astype(np.uint8)
        true_bin = (masks.squeeze(1).float().cpu().numpy() >= 0.50).astype(np.uint8)
        for pred, target in zip(pred_bin, true_bin):
            metrics = compute_segmentation_metrics(pred, target)
            dice_total += metrics["dice"]
            iou_total += metrics["iou"]
            count += 1
    return (
        running_loss / max(1, len(dataloader.dataset)),
        dice_total / max(1, count),
        iou_total / max(1, count),
    )


best_val_dice = -1.0
best_val_iou = 0.0
best_val_loss = float("inf")
best_model_state = None
best_epoch = 0
history = []
run_started = time.perf_counter()
print("[*] Commencing Dense-UNet live training...", flush=True)
for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_dice, val_iou = evaluate_epoch(model, val_loader, criterion)
    if not np.isfinite([train_loss, val_loss, val_dice, val_iou]).all():
        raise FloatingPointError(f"Non-finite epoch metric at epoch {epoch}.")
    scheduler.step()
    epoch_elapsed = time.perf_counter() - epoch_started
    elapsed = time.perf_counter() - run_started
    eta = (elapsed / epoch) * (EPOCHS - epoch)
    images_per_second = (len(train_ds) + len(val_ds)) / max(epoch_elapsed, 1e-9)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "epoch_seconds": epoch_elapsed,
        "images_per_second": images_per_second,
    })
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_val_iou = val_iou
        best_val_loss = val_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        print(
            f"  [*] NEW BEST: epoch={epoch:02d}, Dice={val_dice:.4f}, IoU={val_iou:.4f}",
            flush=True,
        )
    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] | train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | val_dice={val_dice:.4f} | val_iou={val_iou:.4f} | "
        f"speed={images_per_second:.2f} img/s | elapsed={elapsed / 60:.1f} min | "
        f"ETA={eta / 60:.1f} min",
        flush=True,
    )

if best_model_state is None:
    raise RuntimeError("Training ended without a finite best model checkpoint.")
model.load_state_dict(best_model_state, strict=True)
val_loss, val_dice, val_iou = evaluate_epoch(model, val_loader, criterion)
assert abs(val_dice - best_val_dice) < 1e-6, "Restored best checkpoint validation Dice changed."
assert abs(val_iou - best_val_iou) < 1e-6, "Best checkpoint Dice/IoU record is inconsistent."


In [ ]:
# ---------------------------------------------------------------------------
# 5. Checkpoint export, qualitative verification and fail-safe assertion
# ---------------------------------------------------------------------------
import json
import shutil
import matplotlib.pyplot as plt
import pandas as pd

if best_model_state is None or best_epoch < 1:
    raise RuntimeError("No best checkpoint is available for export.")
output_dir = Path("/kaggle/working") if IS_KAGGLE else (REPO / "checkpoints")
output_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "model_state_dict": best_model_state,
    "epoch": int(best_epoch),
    "val_loss": float(best_val_loss),
    "val_dice": float(best_val_dice),
    "val_iou": float(best_val_iou),
    "arch": "DenseUNet-DenseNet121",
    "img_size": int(IMG_SIZE),
    "data_mode": "real" if USE_REAL else "synthetic",
    "num_real_pairs": int(len(PAIRS)),
    "history": history,
}
ckpt_main = output_dir / "dense_unet_best.pth"
ckpt_alias = output_dir / "dense_unet.pth"
torch.save(payload, ckpt_main)
shutil.copy2(ckpt_main, ckpt_alias)
manifest = {key: value for key, value in payload.items() if key not in {"model_state_dict", "history"}}
manifest["training_history_csv"] = str(output_dir / "dense_unet_training_history.csv")
manifest["preview_png"] = str(output_dir / "dense_unet_preview.png")
with (output_dir / "dense_unet_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
pd.DataFrame(history).to_csv(output_dir / "dense_unet_training_history.csv", index=False)
print(f"[OK] Best checkpoint saved: {ckpt_main} ({ckpt_main.stat().st_size / (1024 ** 2):.1f} MB)", flush=True)

model.eval()
with torch.no_grad():
    sample_images, sample_masks = next(iter(val_loader))
    sample_images = sample_images.to(device)
    sample_masks = sample_masks.to(device)
    pred_probs = model(sample_images)

    # Empty-mask fail-safe must bypass gating and preserve input exactly.
    whiteout_mask = torch.zeros_like(pred_probs[:1])
    bypassed = apply_dilated_soft_mask_tensor(sample_images[:1], whiteout_mask)
    assert torch.allclose(bypassed, sample_images[:1], atol=1e-6), "White-out fail-safe bypass violated."
    print("[OK] White-out fail-safe preserved the input scan.", flush=True)

    # Confirm the normal predicted mask is accepted by the live soft-mask pathway.
    predicted_mask = (pred_probs[:1] >= 0.50).float()
    gated = apply_dilated_soft_mask_tensor(sample_images[:1], predicted_mask)
    assert gated.shape == sample_images[:1].shape
    assert torch.isfinite(gated).all(), "Anatomical soft masking produced non-finite pixels."
    print("[OK] Live predicted mask passed the soft-masking smoke check.", flush=True)

image_vis = sample_images[0].cpu().permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN
image_vis = np.clip(image_vis, 0.0, 1.0)
mask_vis = sample_masks[0, 0].cpu().numpy()
pred_vis = pred_probs[0, 0].cpu().numpy()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].imshow(image_vis[..., 0], cmap="gray")
axes[0].set_title("Radiograph")
axes[1].imshow(mask_vis, cmap="gray")
axes[1].set_title("Ground-truth lung mask")
axes[2].imshow(pred_vis, cmap="gray", vmin=0.0, vmax=1.0)
axes[2].set_title("Dense-UNet prediction")
for axis in axes:
    axis.axis("off")
fig.tight_layout()
preview_path = output_dir / "dense_unet_preview.png"
fig.savefig(preview_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[OK] Qualitative preview saved: {preview_path}", flush=True)

print("\n=== STAGE 1 HAND-OFF SUMMARY ===")
print(f"Data mode      : {'REAL' if USE_REAL else 'SYNTHETIC'}")
print(f"Real pairs     : {len(PAIRS)}")
print(f"Best epoch     : {best_epoch}")
print(f"Best val Dice  : {best_val_dice:.4f}")
print(f"Best val IoU   : {best_val_iou:.4f}")
print(f"Output folder  : {output_dir}")

# Remove only notebook/repository temporary directories; preserve exported files.
if IS_KAGGLE:
    stale_dirs = [
        WORK / "Chest-X-rays",  # Legacy clone location from older notebook versions.
        WORK / "LungRay",       # Legacy alternate clone location.
        WORK / "__results___files",
        *WORK.glob(".lungray_dense_unet_tmp_*"),
    ]
    for temp_dir in stale_dirs:
        if temp_dir.is_dir():
            resolved = temp_dir.resolve()
            if resolved.parent == WORK.resolve():
                shutil.rmtree(resolved)
                print(f"[*] Removed temporary directory: {resolved}", flush=True)


## Stage 1 completion criteria
- The Dense-UNet checkpoint is written to `checkpoints/dense_unet.pth` (and a `dense_unet_best.pth` alias).
- GPU-dilated soft masking plus the White-out Lung Fail-Safe are asserted on live model output.
- No bounding boxes are used at any point: this stage trains only from pixel-level lung masks.
- Notebook 09 consumes this checkpoint as a strictly frozen anatomical gate (`requires_grad = False`).
